# Head label dengan pembobotan kelas
Label manual tetap ground truth. Bobot dihitung hanya dari section layak pada train.
Validasi tetap tanpa bobot dan memakai split/konteks asli. Tidak membaca test.
Bandingkan macro F1 dengan worship-v3 (0.7237543587). Eksperimen ini mulai dari
pretrained dengan konfigurasi head-only asli; satu perubahan utama: pembobotan loss.


In [ ]:
import json, sys, subprocess
from pathlib import Path
from datetime import datetime
assert Path('finetuning/config.json').is_file(), 'Gunakan root Section-Annotations'
cfg = json.loads(Path('finetuning/config.json').read_text())
cfg['training'].update(trainable='function_head', learning_rate=1e-4,
    window_seconds=None, gradient_checkpointing=False, class_weighting='sqrt_inverse')
config_path = Path('outputs.local/weighted-head-config.json')
config_path.parent.mkdir(exist_ok=True)
config_path.write_text(json.dumps(cfg, indent=2))
print(sys.executable)
print(json.dumps(cfg['training'], indent=2))


Bobot dan jumlah section tiap kelas akan tampil dan disimpan dalam run.json. Lolos smoke tidak menjamin seluruh run muat, tetapi head-only sebelumnya sudah berjalan pada Mac ini.

In [ ]:
result = subprocess.run([sys.executable, '-u', '-m', 'finetuning.train',
    '--config', str(config_path), '--device', 'mps', '--smoke-test'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(result.stdout)
result.check_returncode()


In [ ]:
run_name = 'worship-weighted-' + datetime.now().strftime('%Y%m%d-%H%M%S')
log_path = Path('outputs.local') / f'{run_name}.log'
command = [sys.executable, '-u', '-m', 'finetuning.train',
    '--config', str(config_path), '--device', 'mps', '--run-name', run_name]
print('Run:', run_name, '| Log:', log_path)
with log_path.open('w') as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        code = process.wait()
    except KeyboardInterrupt:
        process.terminate()
        process.wait()
        raise
if code:
    raise RuntimeError(f'Training gagal: {log_path}')


Bandingkan checkpoint terbaik termasuk baseline epoch 0. Loss berbobot tidak dapat dibandingkan langsung dengan loss run lama. F1 validasi tetap dapat dibandingkan. Jangan ubah anotasi atau split untuk mengejar nilai.

In [ ]:
import torch
run_dir = Path(cfg['work_dir']) / 'runs' / run_name
checkpoint = torch.load(run_dir / 'best.pt', map_location='cpu', weights_only=False)
f1 = checkpoint['validation']['macro_f1_present']
reference = json.loads(Path('outputs.local/val-worship-v3.json').read_text())['macro_f1_present']
print('Epoch terbaik:', checkpoint['epoch'])
print(f'Weighted: {f1:.6f}; worship-v3: {reference:.6f}; delta: {f1-reference:+.6f}')
print('Nilai lebih tinggi perlu dilihat bersama akurasi dan F1 per kelas; belum bukti signifikansi statistik.')
